In [22]:
import requests
import torch
import re
import time
import psutil
import subprocess

import pandas as pd

from datasets import load_dataset

In [23]:
ds = load_dataset("cardiffnlp/tweet_eval", "irony")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 784 entries, 0 to 783
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    784 non-null    object
 1   label   784 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 12.4+ KB


In [24]:
test['label'] = test['label'].apply(lambda x: 'ironic' if x == 1 else 'sincere')

labels = test['label'].unique()

test

,text,label
0,@user Can U Help?||More conservatives needed o...,sincere
1,"Just walked in to #Starbucks and asked for a ""...",ironic
2,#NOT GONNA WIN,sincere
3,@user He is exactly that sort of person. Weirdo!,sincere
4,So much #sarcasm at work mate 10/10 #boring 10...,ironic
...,...,...
779,"If you drag yesterday into today, your tomorro...",sincere
780,Congrats to my fav @user & her team & my birth...,sincere
781,@user Jessica sheds tears at her fan signing e...,sincere
782,#Irony: al jazeera is pro Anti - #GamerGate be...,ironic


In [25]:
def get_ollama_memory_usage(port=11434):
    """
    Finds the process listening on the given port using psutil
    and returns its memory usage in bytes (RSS).
    Returns None if the process isn't found or can't be accessed.
    """
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            # Call proc.connections() to see if it's listening on the desired port
            for conn in proc.connections(kind='inet'):
                if conn.laddr.port == port:
                    # Found the process that listens on port=11434
                    memory_info = proc.memory_info()
                    return memory_info.rss  # in bytes
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            pass
    
    # If no process was found
    return None

get_ollama_memory_usage()

C:\Users\Rafael\AppData\Local\Temp\ipykernel_16184\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


98099200

In [26]:
def get_gpu_memory_usage():
    """
    Returns a list of used memory (in MB) for each GPU.
    """
    # Use nvidia-smi with the --query-gpu and --format flags to get just the memory usage
    command = [
        "nvidia-smi",
        "--query-gpu=memory.used",  # You can also add memory.free, name, etc.
        "--format=csv,noheader,nounits"  # CSV output with no header or units
    ]
    try:
        output = subprocess.check_output(command)
        # Decode the output from bytes to string
        output_str = output.decode("utf-8").strip()
        # Each line corresponds to one GPU's memory usage
        usage_values = [int(x) for x in output_str.split("\n")]
        return usage_values[0]
    except subprocess.CalledProcessError as e:
        print("Error running nvidia-smi:", e)
        return []


In [27]:
def classify(text, labels):
    url = "http://localhost:11434/api/chat"
    
    messages = [
        {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling binary classification tasks based on user instructions."},
        {"role": "user", "content": f"Classify the following text based on the task: Sentiment analysis of tweets. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"}
    ]
    
    start_time = time.time()
    response = requests.post(url, json={
        "model": "deepseek-r1:1.5b",
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": 0,
            "num_predict": 3100
        }
    })
    response_time = time.time() - start_time
    vram_usage = get_gpu_memory_usage()
    ram_usage_bytes = get_ollama_memory_usage(port=11434) / (1024 * 1024)
    response = response.json()
    response_text = response['message']['content']
    content = response_text.lower()
    
    if '</think>' in content:
        classification_text = content.split('</think>')[1].strip()
        total_time = response['total_duration'] / 1_000_000_000
    else:
        messages.append({"role": "assistant", "content": content + '</think>'})
        start_time2 = time.time()
        response2 = requests.post(url, json={
            "model": "deepseek-r1:1.5b",
            "messages": messages,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 430
            }
        })
        response_time += time.time() - start_time2
        response2 = response2.json()
        classification_text = response2['message']['content'].lower()
        total_time = response['total_duration'] / 1_000_000_000 + response2['total_duration'] / 1_000_000_000
    
    label_counts = {label: len(re.findall(r'\b' + re.escape(label.lower()) + r'\b', classification_text)) for label in labels}
    
    if all(count == label_counts[labels[0]] for count in label_counts.values()):
        content = 'error'
    else:
        content = max(label_counts, key=label_counts.get)
    
    print(f"Text: {text}")
    print(f"Response: {content}")
    
    return content, label_counts, response_time, vram_usage, ram_usage_bytes, total_time, response_text

In [28]:
# apply the classify function to the test set. create one column for each output
test[['prediction', 'label_counts','response_time', 'vram_usage', 'ram_usage', 'total_time', 'response_text']] = test['text'].apply(lambda x: classify(x, labels)).apply(pd.Series)

C:\Users\Rafael\AppData\Local\Temp\ipykernel_16184\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


Text: @user Can U Help?||More conservatives needed on #TSU + get paid 4 posting stuff like this!||YOU $ can go to
Response: ironic
Text: Just walked in to #Starbucks and asked for a "tall blonde" Hahahaha #irony
Response: ironic
Text: #NOT GONNA WIN
Response: ironic
Text: @user He is exactly that sort of person. Weirdo!
Response: ironic
Text: So much #sarcasm at work mate 10/10 #boring 100% #dead mate full on #shit absolutely #sleeping mate can't handle the #sarcasm
Response: ironic
Text: Corny jokes are my absolute favorite
Response: ironic
Text: People complain about my backround pic and all I feel is like "hey don't blame me, Albert E might have spoken those words" #sarcasm #life
Response: sincere
Text: @user @user Darn, my sock joke needs fixing?
Response: sincere
Text: if Christian expects Fifa to sleep in my bed with me tonight, he's wrong 👿
Response: sincere
Text: People who tell people with anxiety to "just stop worrying about it" are my favorite kind of people #not #educateyou

In [29]:
test.to_csv('results/deepseekR1_ZS_binary3.csv', index=False)
test

,text,label,prediction,label_counts,response_time,vram_usage,ram_usage,total_time,response_text
0,@user Can U Help?||More conservatives needed o...,sincere,ironic,"{'sincere': 0, 'ironic': 1}",6.207104,2503,93.085938,4.165240,"<think>\nOkay, so I need to classify the given..."
1,"Just walked in to #Starbucks and asked for a ""...",ironic,ironic,"{'sincere': 0, 'ironic': 1}",4.910418,2512,93.582031,2.856907,"<think>\nAlright, let's tackle this classifica..."
2,#NOT GONNA WIN,sincere,ironic,"{'sincere': 0, 'ironic': 1}",7.127988,2509,93.582031,5.090886,"<think>\nAlright, let's tackle this query step..."
3,@user He is exactly that sort of person. Weirdo!,sincere,ironic,"{'sincere': 0, 'ironic': 1}",6.101017,2504,93.722656,4.066868,"<think>\nOkay, so I need to figure out how to ..."
4,So much #sarcasm at work mate 10/10 #boring 10...,ironic,ironic,"{'sincere': 0, 'ironic': 1}",5.640712,2504,93.726562,3.598365,"<think>\nAlright, let's break this down step b..."
...,...,...,...,...,...,...,...,...,...
779,"If you drag yesterday into today, your tomorro...",sincere,error,"{'sincere': 0, 'ironic': 0}",4.756420,2527,67.828125,2.706686,"<think>\nAlright, let's see here. I need to cl..."
780,Congrats to my fav @user & her team & my birth...,sincere,sincere,"{'sincere': 1, 'ironic': 0}",3.897944,2527,67.832031,1.850853,"<think>\nAlright, so I need to classify this t..."
781,@user Jessica sheds tears at her fan signing e...,sincere,sincere,"{'sincere': 2, 'ironic': 0}",5.749963,2527,68.335938,3.714371,"<think>\nAlright, so I need to figure out how ..."
782,#Irony: al jazeera is pro Anti - #GamerGate be...,ironic,error,"{'sincere': 0, 'ironic': 0}",6.917419,2527,68.406250,4.879761,"<think>\nAlright, let's tackle this classifica..."


In [30]:
y_pred = test['prediction']
y_true = test['label']

#import acc, f1_score, precision and recall from sklearn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.538265
F1 score: 0.581093
Precision: 0.640215
Recall: 0.538265


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [31]:
# get average response time, vram usage and ram usage
response_time_avg = test['response_time'].mean()
vram_usage_avg = test['vram_usage'].mean()
ram_usage_avg = test['ram_usage'].mean()
total_time_avg = test['total_time'].mean()

print(f'Average response time: {response_time_avg}')
print(f'Average VRAM usage: {vram_usage_avg}')
print(f'Average RAM usage: {ram_usage_avg}')
print(f'Average total time: {total_time_avg}')

Average response time: 5.901179294501032
Average VRAM usage: 2525.276785714286
Average RAM usage: 83.3271783322704
Average total time: 3.86111508252551


In [32]:
# save results to txt
with open('results/deepseekR1_ZS_binary3.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {response_time_avg}\n')
    f.write(f'Average VRAM usage: {vram_usage_avg}\n')
    f.write(f'Average RAM usage: {ram_usage_avg}\n')
    f.write(f'Average total time: {total_time_avg}\n')
    f.write(f'Lines classified: {len(test)}\n')
    